## Middleware


In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")


## Summarization model

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

### Messagebased summarization middleware

agent = create_agent(
    model="claude-sonnet-4-6",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="claude-sonnet-4-6",
        trigger=("messages", 10),
        keep=("messages", 4)
    )],
)

In [3]:
## Run with thread Id
config={"configurable": {"thread_id": "test-1"}}


In [7]:
query = ["what is 2+2?", "what is 3+3?", "what is 4+4?", "what is 5+5?", "what is 6+6?", "what is 7+7?", "what is 8+8?", "what is 9+9?", "what is 10+10?", "what is 11+11?"]

for q in query:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )
    print(response)
    print(len(response["messages"]))

{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='25b627e2-4cfa-4e19-aeb0-ead1b661e179'), AIMessage(content='2 + 2 = **4**', additional_kwargs={}, response_metadata={'id': 'msg_011Cg8TX9UDxubUzbk7yd5RK', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 14, 'output_tokens': 14, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'diagnostics': None, 'model_name': 'claude-sonnet-4-6', 'model_provider': 'anthropic'}, id='lc_run--01a14b99-6677-7f03-8087-80270f4ef2b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 14, 'total_tokens': 28, 'input_token_details': {'cache_creatio

### Token Size 

In [16]:

from langchain_community.tools import tool


@tool
def search_hotels(city:str)-> str:
    """
    Search for hotels in a given city.
    Args:
        city (str): The name of the city to search hotels in.
    Returns:
        str: A list of hotels in the specified city.
    """
    # Simulated hotel search results
    hotels = {
        "New York": ["Hotel A", "Hotel B", "Hotel C"],
        "Los Angeles": ["Hotel D", "Hotel E", "Hotel F"],
        "Chicago": ["Hotel G", "Hotel H", "Hotel I"]
    }
    
    return hotels.get(city, ["No hotels found in this city."])



agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="claude-sonnet-4-6",
        trigger=("tokens", 510),
        keep=("tokens", 200)
    )],
)

cofig={"configurable": {"thread_id": "test-2"}}

def count_tokens(messages):
    # Simulated token counting logic
    return sum(len(str(message.content)) for message in messages)

In [2]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="claude-sonnet-4-6",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="claude-sonnet-4-6",
            trigger=("tokens",1000),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [22]:
cities=["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "India", "London", "Paris", "Berlin", "Tokyo"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search for hotels in {city}.")]},
        config=config
    )
    token_count = count_tokens(response["messages"])
    print(f"{city}: ~{token_count} tokens, {len(response['messages'])} messages")
    print(response["messages"])

New York: ~266 tokens, 4 messages
[HumanMessage(content='Search for hotels in New York.', additional_kwargs={}, response_metadata={}, id='6551878d-e457-43aa-8360-7c466097380f'), AIMessage(content=[{'id': 'toolu_013QRKc4BJmyf14Q41k6NaMK', 'caller': {'type': 'direct'}, 'input': {'city': 'New York'}, 'name': 'search_hotels', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cg8VtLbmqtBTEFMHVT9JT', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 569, 'output_tokens': 55, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'diagnostics': None, 'model_name': 'claude-sonnet-4-6', 'model_provider': 'anthropic'}, id='lc_run-

### Fraction Summarization Middleware

In [3]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="claude-sonnet-4-6",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="claude-sonnet-4-6",
            trigger=("fraction",0.005), # 0.5% = ~640 tokens of the total tokens in the conversation
            keep=("fraction",0.002),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [28]:
cities=["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "India", "London", "Paris", "Berlin", "Tokyo"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search for hotels in {city}.")]},
        config=config
    )
    token_count = count_tokens(response["messages"])
    fraction = token_count / 128000
    print(f"{city}: ~{token_count} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(response["messages"])

New York: ~305 tokens (0.2383%), 4 messages
[HumanMessage(content='Search for hotels in New York.', additional_kwargs={}, response_metadata={}, id='a2c88edf-2fd5-4ddc-a0f9-dbeec5c4c14b'), AIMessage(content=[{'text': 'Let me search for hotels in New York for you right away!', 'type': 'text'}, {'id': 'toolu_0139bdJEVh4wTuEpNARZv1Wz', 'caller': {'type': 'direct'}, 'input': {'city': 'New York'}, 'name': 'search_hotels', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cg8WWx3z5VDwes13Z8aCs', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 569, 'output_tokens': 68, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'di

## Human In the loop Middleware

In [17]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def read_email_tool(email_id: str) -> str:
    """Simulated tool to read an email by ID."""
    return f"Email content for ID {email_id}: This is a simulated email content."

@tool
def send_email_tool(email_id: str, content: str) -> str:
    """Simulated tool to send an email by ID."""
    return f"Email with ID {email_id} sent with content: {content}"

In [18]:
agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        ),
    ]
)

In [19]:
config = {"configurable": {"thread_id": "test-3"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Please send an email using send_email_tool with email_id '123' and content 'Hello, this is a test email.'")]},
    config=config
)


In [34]:
response

{'messages': [HumanMessage(content="Please send an email using send_email_tool with email_id '123' and content 'Hello, this is a test email.'", additional_kwargs={}, response_metadata={}, id='33c83b85-258b-4b4b-a926-91db1ffa5479'),
  AIMessage(content=[{'text': "I'll send that email right away for you!", 'type': 'text'}, {'id': 'toolu_01NzWk7cJUHYs7eWcpu3g1zN', 'caller': {'type': 'direct'}, 'input': {'email_id': '123', 'content': 'Hello, this is a test email.'}, 'name': 'send_email_tool', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cg8Y4DNi5y8mTYdXtr8ue', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 674, 'output_tokens': 91, 'output_tokens_details': None, 'ser

In [20]:
from langgraph.types import Command

if "__interrupt__" in response:
    print("Human-in-the-loop intervention required.")

    result= agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {response['messages'][-1].content}")

Human-in-the-loop intervention required.
Result: [{'id': 'toolu_018kvnRYE7FrqVwezdf8GunT', 'caller': {'type': 'direct'}, 'input': {'email_id': '123', 'content': 'Hello, this is a test email.'}, 'name': 'send_email_tool', 'type': 'tool_use', 'toolset_name': None}]


### REJECT

In [21]:
config = {"configurable": {"thread_id": "test-3"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="send email john@gmail.com with the subject 'how are you' and body 'I hope you are doing well.'")]},
    config=config
)

In [10]:
response

{'messages': [HumanMessage(content="Please send an email using send_email_tool with email_id '123' and content 'Hello, this is a test email.'", additional_kwargs={}, response_metadata={}, id='18a9ed94-269f-4b3d-851e-631d69e4a055'),
  AIMessage(content=[{'text': "I'll send that email right away for you!", 'type': 'text'}, {'id': 'toolu_01LkebvHJ7uW3RfFtrZnidpU', 'caller': {'type': 'direct'}, 'input': {'email_id': '123', 'content': 'Hello, this is a test email.'}, 'name': 'send_email_tool', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgB75qiNBQLzJJg5d6ZBQ', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 674, 'output_tokens': 91, 'output_tokens_details': None, 'ser

In [23]:
from langgraph.types import Command

if "__interrupt__" in response:
    print("Paused! Rejecting the action.")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print("Resumed result:")
    print(result["messages"][-1].content)

In [24]:
from langgraph.types import Command

print("Initial response:")
print(response)

# Check for interrupts in the LangGraph response
if response.get("__interrupt__"):
    print("\nPaused! Rejecting the action.")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print("\nFull resumed result:")
    print(result)

    print("\nMessages:")
    for msg in result.get("messages", []):
        print(f"\nRole: {msg.type}")
        print(f"Content: {msg.content}")
else:
    print("No interrupt detected in response.")

Initial response:
{'messages': [HumanMessage(content="Please send an email using send_email_tool with email_id '123' and content 'Hello, this is a test email.'", additional_kwargs={}, response_metadata={}, id='a6be4150-e55f-4630-bdb6-2940282a6a8a'), AIMessage(content=[{'id': 'toolu_018kvnRYE7FrqVwezdf8GunT', 'caller': {'type': 'direct'}, 'input': {'email_id': '123', 'content': 'Hello, this is a test email.'}, 'name': 'send_email_tool', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgB7wV4YZTwExYSKffVh2', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 674, 'output_tokens': 81, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'spee